## QRS area vs age

* Before running this notebook, download all the data files from zenodo
* Run the script `scripts/one_beat.py` before running this notebook

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_DIR = "../data"

In [ ]:
summary_area = []
for n in range(1, 17):
    df_summary = pd.read_csv(
        f"{DATA_DIR}/peak_summary_{n}.csv",
        usecols=['subject', 'exam_id', 'hr', 'retain_subject'],
    )
    df_summary = df_summary[df_summary['retain_subject']].reset_index(drop=True)
    df_summary = df_summary.groupby('subject')[['exam_id', 'hr']].first().reset_index()
    data_array = np.load(f"{DATA_DIR}/one_beat_array_{n}.npy")

    start = 2040
    end = 2060
    channel = [x for x in range(12)]
    replace_areas = []

    for i in range(len(data_array)):
        area = []
        for chan in channel:
            arange_start = float(data_array[i, start, chan])
            arange_end = float(data_array[i, end - 1,  chan])
            try:
                replace_val = np.linspace(arange_start, arange_end, (end - start))
            except ZeroDivisionError:
                # some subjects have faster heartbeats; so there will be zero padding
                replace_val = 0
            area.append(float(np.sum(np.abs(data_array[i, start:end, chan] - replace_val))))
        replace_areas.append(area)
    replace_areas = pd.DataFrame(replace_areas, columns=[f'chan_{x}' for x in range(12)])

    df_summary = df_summary.merge(
        replace_areas,
        left_on=df_summary.index,
        right_on=replace_areas.index,
    ).drop(columns=['key_0'])
    summary_area.append(df_summary)

In [ ]:
df_meta = pd.read_csv(
    f"{DATA_DIR}/exams.csv",
    usecols=['exam_id', 'age', 'is_male', 'nn_predicted_age', 'normal_ecg']
)
df_summary = pd.concat(summary_area, axis=0).reset_index(drop=True)
df = df_summary.merge(
    df_meta,
    on='exam_id'
)
df.shape

In [ ]:
df.head()

In [ ]:
plt.scatter(
    df['age'],
    df['chan_0']
)
plt.title("All Subject with Certain Filters")
plt.xlabel("Age")
plt.ylabel("Area Replaced Chan 0 (mV)")
plt.show()

In [ ]:
neutral_age_filter = abs(df['age'] - df['nn_predicted_age']) < 8

plt.scatter(
    df[neutral_age_filter]['age'],
    df[neutral_age_filter]['chan_0']
)
plt.xlim(15, 100)
plt.ylim(0, 60)
plt.title("Neutral Predicted Age")
plt.xlabel("Age")
plt.ylabel("Area Replaced Chan 0 (mV)")
plt.show()

In [ ]:
neutral_age_filter = abs(df['age'] - df['nn_predicted_age']) < 8
normal_ecg = df['normal_ecg']
old_ecg = df['nn_predicted_age'] - df['age'] >= 8
young_ecg = df['age'] - df['nn_predicted_age'] >= 8

plt.scatter(
    df[(neutral_age_filter) & (normal_ecg)]['age'],
    df[(neutral_age_filter) & (normal_ecg)]['chan_0']
)
plt.xlim(15, 100)
plt.ylim(0, 60)
plt.title("Neutral Predicted Age")
plt.xlabel("Age")
plt.ylabel("Area Replaced Chan 0 (mV)")
plt.show()

In [ ]:
plt.scatter(
    df[(young_ecg) & (normal_ecg)]['age'],
    df[(young_ecg) & (normal_ecg)]['chan_0']
)
plt.xlim(15, 100)
plt.ylim(0, 60)
plt.title("Young Predicted Age")
plt.xlabel("Age")
plt.ylabel("Area Replaced Chan 0 (mV)")
plt.show()

In [ ]:
plt.scatter(
    df[(old_ecg) & (normal_ecg)]['age'],
    df[(old_ecg) & (normal_ecg)]['chan_0']
)
plt.xlim(15, 100)
plt.ylim(0, 60)
plt.title("Old Predicted Age")
plt.xlabel("Age")
plt.ylabel("Area Replaced Chan 0 (mV)")
plt.show()

In [ ]:
plt.scatter(
    df[(young_ecg) & (normal_ecg)]['age'],
    df[(young_ecg) & (normal_ecg)]['chan_0'],
    alpha=0.1,
    c='g',
    label='Yound ECG'
)
plt.scatter(
    df[(old_ecg) & (normal_ecg)]['age'],
    df[(old_ecg) & (normal_ecg)]['chan_0'],
    alpha=0.1,
    c='r',
    label='Old ECG'
)
plt.xlim(15, 100)
plt.ylim(0, 60)
plt.title("Young-Old Predicted Age; Normal ECG")
plt.xlabel("Age")
plt.ylabel("Area Replaced Chan 0 (mV)")
plt.legend()
plt.show()

In [ ]:
channel_cols = [col for col in df.columns if 'chan_' in col]
df.loc[:, "all_chan_area"] = df[channel_cols].sum(axis=1)

plt.scatter(
    df[(young_ecg) & (normal_ecg)]['age'],
    df[(young_ecg) & (normal_ecg)]['all_chan_area'],
    alpha=0.1,
    c='g',
    label='Yound ECG'
)
plt.scatter(
    df[(old_ecg) & (normal_ecg)]['age'],
    df[(old_ecg) & (normal_ecg)]['all_chan_area'],
    alpha=0.1,
    c='r',
    label='Old ECG'
)
plt.xlim(15, 100)
# plt.ylim(0, 60)
plt.title("Young-Old Predicted Age; Normal ECG")
plt.xlabel("Age")
plt.ylabel("Area Replaced All Chan (mV)")
plt.legend()
plt.show()

In [ ]:

plt.scatter(
    df[(young_ecg) & (normal_ecg)]['age'],
    df[(young_ecg) & (normal_ecg)]['all_chan_area'],
    alpha=0.1,
    c='g',
    label='Yound ECG'
)
plt.scatter(
    df[(old_ecg) & (normal_ecg)]['age'],
    df[(old_ecg) & (normal_ecg)]['all_chan_area'],
    alpha=0.1,
    c='r',
    label='Old ECG'
)
plt.xlim(15, 100)
# plt.ylim(0, 60)
plt.title("Young-Old Predicted Age; Normal ECG")
plt.xlabel("Age")
plt.ylabel("Area Replaced All Chan (mV)")
plt.legend()
plt.show()

In [ ]:
df[young_ecg]

In [ ]:
df.loc[:, 'classification'] = "Neutral"
df.loc[young_ecg, 'classification'] = 'Young'
df.loc[old_ecg, 'classification'] = 'Old'

In [ ]:
df_grp = df.groupby(
    ['age', 'classification', 'normal_ecg', 'is_male']
)['all_chan_area'].mean().reset_index()

df_class = df.groupby(
    ['age', 'classification']
).agg(
    area_mean=('all_chan_area', np.mean),
    area_std=('all_chan_area', np.std)
).reset_index()

In [ ]:
plt.scatter(
    df_grp[df_grp['normal_ecg']]['age'],
    df_grp[df_grp['normal_ecg']]['all_chan_area'],
)

In [ ]:
# scatter plot age vs area for each group, classification, normal_ecg, is_male

young_ecg = df_class['classification'] == 'Young'
old_ecg = df_class['classification'] == 'Old'
neutral_ecg = df_class['classification'] == 'Neutral'

plt.scatter(
    df_class[young_ecg]['age'],
    df_class[young_ecg]['area_mean'],
    c='g',
    alpha=0.5,
    label='Young ECG'
)

plt.scatter(
    df_class[neutral_ecg]['age'],
    df_class[neutral_ecg]['area_mean'],
    c='b',
    alpha=0.5,
    label='Neutral ECG'
)

plt.scatter(
    df_class[old_ecg]['age'],
    df_class[old_ecg]['area_mean'],
    c='r',
    alpha=0.5,
    label='Old ECG'
)
plt.xlabel('Age')
plt.ylabel('Mean Area Replaced')
plt.legend()
plt.show()

In [ ]:
# scatter plot age vs area for each group, classification, normal_ecg, is_male

young_ecg = df_class['classification'] == 'Young'
old_ecg = df_class['classification'] == 'Old'
neutral_ecg = df_class['classification'] == 'Neutral'

plt.scatter(
    df_class[young_ecg]['age'],
    df_class[young_ecg]['area_mean'],
    c='g',
    alpha=0.5,
    label='Young ECG'
)

plt.scatter(
    df_class[neutral_ecg]['age'],
    df_class[neutral_ecg]['area_mean'],
    c='b',
    alpha=0.5,
    label='Neutral ECG'
)

plt.scatter(
    df_class[old_ecg]['age'],
    df_class[old_ecg]['area_mean'],
    c='r',
    alpha=0.5,
    label='Old ECG'
)
plt.ylim(100, 202)
plt.xlabel('Age')
plt.ylabel('Mean Area Replaced')
plt.legend()
plt.show()

In [ ]:
df_gender = df.groupby(
    ['age', 'is_male']
).agg(
    area_mean=('all_chan_area', np.mean),
    area_std=('all_chan_area', np.std)
).reset_index()

male_filt = df_gender['is_male']
plt.scatter(
    df_gender[~male_filt]['age'],
    df_gender[~male_filt]['area_mean'],
    c='coral',
    alpha=0.5,
    label='Female'
)

plt.scatter(
    df_gender[male_filt]['age'],
    df_gender[male_filt]['area_mean'],
    c='dodgerblue',
    alpha=0.5,
    label='Male'
)
plt.xlabel('Age')
plt.ylabel('Mean Area Replaced')
plt.legend()
plt.show()

In [ ]:
df_normal = df.groupby(
    ['age', 'normal_ecg']
).agg(
    area_mean=('all_chan_area', np.mean),
    area_std=('all_chan_area', np.std)
).reset_index()

normal_filt = df_normal['normal_ecg']
plt.scatter(
    df_normal[~normal_filt]['age'],
    df_normal[~normal_filt]['area_mean'],
    c='crimson',
    alpha=0.5,
    label='Not Normal'
)
plt.scatter(
    df_normal[normal_filt]['age'],
    df_normal[normal_filt]['area_mean'],
    c='darkolivegreen',
    alpha=0.5,
    label='Normal'
)
plt.xlabel('Age')
plt.ylabel('Mean Area Replaced')
plt.legend()
plt.show()